# Long-running Training + Asynchronous Evals — HiLift edition

**The problem.** Real production training runs for hours or days. Real
evaluation batteries (held-out benchmarks, per-slice metrics, latency
profiling) take minutes to hours **per checkpoint**. You cannot afford
to inline them into the training loop, and you cannot afford to
serialise "train an epoch, then eval that checkpoint, then train the
next epoch".

**The anti-pattern.** Reopening the training run from an eval process
(`wandb.init(id=train_run_id, resume="allow")`) is unsafe — it causes
config overwriting, step interleaving, `run.finish()` races, and
summary last-write-wins. See
`notebooks/advanced/asynchronous_evals/ARCHITECTURE.md` for the
detailed treatment.

**The safe pattern.** *One job = one run*, tied together by a shared
`group=training_run_id` and by artifact lineage. This notebook
demonstrates it end-to-end on the workshop's HiLift Transolver++
surrogate:

1. **§3 — Find checkpoints.** The training notebook
   `experiment_tracking_hilift.ipynb` already produced a
   `hilift-transolverpp-train` run whose logged artifacts include
   ~5–7 checkpoint versions (epochs 10, 20, 30, 40, 50 plus any
   `is_best` carve-outs). This notebook queries that run and
   iterates its checkpoints — no training happens here.
2. **§4 — N async eval runs.** For each checkpoint, a separate
   `wandb.init(job_type="eval", group=training_run_id, resume="never")`
   opens its own run, `use_artifact`s the checkpoint (creating the
   lineage edge), runs the offline eval battery (global RMSE / MAE
   on Cp; per-AoA and per-geometry buckets; latency), and logs an
   eval-results artifact. In production each of these would fire on
   a different machine — here they run sequentially in one kernel;
   the API is identical either way.
3. **§5 — Aggregation.** After the eval runs finish, the API
   discovers them by server-side filter on
   `config.source_training_run_id`, computes best / latest, and
   writes compact `eval/*` keys back into the training run's
   summary via `api.run(...).summary.update({...})`. Namespaced
   under `eval/*` — nothing training-owned is overwritten.

**Prerequisites.**
- `registry.ipynb` executed → dataset at
  `wandb-registry-hilift/pointcloud_demo:production`.
- `experiment_tracking_hilift.ipynb` executed → a
  `hilift-transolverpp-train` run in `wandb_emea/jb_physics_demo`
  with checkpoint artifacts on it.
- Internet — §2 clones `thuml/Transolver_plus` if the training
  notebook hasn't already cached it.


## 1. Setup

Third-party imports and section-local constants. `TRAINING_RUN_QUERY`
names the training run this eval fleet consumes — every eval run
below groups by that run's ID.


In [ ]:
from __future__ import annotations

import importlib
import json
import subprocess
import sys
import tempfile
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

import wandb

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def utc_ts() -> str:
    return datetime.now(timezone.utc).isoformat()


In [ ]:
# ---- W&B target ----
ENTITY = "wandb_emea"
PROJECT = "jb_physics_demo"

# ---- Registry paths ----
REGISTRY_NAME = "hilift"
DATASET_COLLECTION = "pointcloud_demo"
DATASET_REGISTRY_PATH = f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}:production"

# ---- The training run this notebook evaluates ----
# Set by experiment_tracking_hilift.ipynb §4c. If you renamed the
# training run, change this string.
TRAINING_RUN_QUERY = "hilift-transolverpp-train"

# ---- Chunking for full-case prediction ----
CHUNK_POINTS = 4096

TMP_DIR = Path(tempfile.mkdtemp(prefix="hilift_async_eval_"))

print(f"torch    = {torch.__version__}")
print(f"wandb    = {wandb.__version__}")
print(f"device   = {device}")
print(f"tmp      = {TMP_DIR}")
print(f"training = {ENTITY}/{PROJECT}   run name: {TRAINING_RUN_QUERY}")


## 2. Model + data — reused from the training notebook

Every eval run needs to be able to rebuild the model architecture,
load a state_dict, and score full cases from the Registry dataset.
The helpers below are lifted from `experiment_tracking_hilift.ipynb`
(§2, §3a, §3b, §4a) — same clone / patch of `thuml/Transolver_plus`,
same `TransolverPointFieldSurrogate` wrapper, same NPZ loader and
case splitter. `predict_case` is parameterised on `model` (rather
than reading a module-level global as in the training notebook) so
each eval iteration can call it with a freshly-loaded checkpoint.

Eval uses the checkpoint's *embedded* stats (packed into every
checkpoint dict by the training notebook) — not stats re-derived
here — so the normalisation matches what the model was trained on.
The `stats` derived below is only used for `aoa_condition`, which
depends on AoA mean/std that are stable across the workshop.


In [ ]:
# ==== §2 — Transolver++ clone/patch + model wrapper + data helpers ===

# ---- Transolver++ clone + patch (reused cache if the training notebook already ran) ----
CACHE_ROOT = Path.home() / ".cache" / "physicsx_demo"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
TRANSOLVER_DIR = CACHE_ROOT / "Transolver_plus"

if not TRANSOLVER_DIR.exists():
    print(f"Cloning thuml/Transolver_plus into {TRANSOLVER_DIR} ...")
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/thuml/Transolver_plus.git",
        str(TRANSOLVER_DIR),
    ])
else:
    print(f"Using cached Transolver_plus at {TRANSOLVER_DIR}")

model_file = TRANSOLVER_DIR / "models" / "Transolver_plus.py"
src_text = model_file.read_text()
if "def _maybe_all_reduce" not in src_text:
    src_text = src_text.replace(
        "import torch.distributed.nn as dist_nn\n",
        "import torch.distributed.nn as dist_nn\n"
        "import torch.distributed as dist\n\n\n"
        "def _maybe_all_reduce(tensor):\n"
        "    if dist.is_available() and dist.is_initialized() "
        "and dist.get_world_size() > 1:\n"
        "        dist_nn.all_reduce(tensor, op=dist_nn.ReduceOp.SUM)\n"
        "    return tensor\n",
    )
    src_text = src_text.replace(
        "dist_nn.all_reduce(slice_norm, op=dist_nn.ReduceOp.SUM)",
        "_maybe_all_reduce(slice_norm)",
    )
    src_text = src_text.replace(
        "dist_nn.all_reduce(slice_token, op=dist_nn.ReduceOp.SUM)",
        "_maybe_all_reduce(slice_token)",
    )
    model_file.write_text(src_text)
    print("Patched Transolver_plus.py with _maybe_all_reduce.")
else:
    print("Transolver_plus.py already patched.")

if str(TRANSOLVER_DIR) not in sys.path:
    sys.path.insert(0, str(TRANSOLVER_DIR))
if "models.Transolver_plus" in sys.modules:
    importlib.reload(sys.modules["models.Transolver_plus"])
from models.Transolver_plus import Model as TransolverPlusModel


# ---- Data helpers (same signatures as the training notebook) ----
def _load_npz_case(path: Path) -> dict:
    """Load one setup-notebook NPZ back into a plain-Python dict."""
    data = np.load(path, allow_pickle=True)
    case = {k: data[k] for k in data.files}
    for key in ["geo_id", "run_id", "target_name",
                "target_location", "target_mode"]:
        if key in case:
            case[key] = (
                str(case[key].item()) if case[key].ndim == 0 else str(case[key])
            )
    case["aoa"] = float(np.asarray(case["aoa"]).item())
    return case


def aoa_condition(aoa_degrees: float, stats: dict) -> np.ndarray:
    """Encode angle of attack as (z-norm, sin(rad), cos(rad))."""
    aoa_norm = (np.float32(aoa_degrees) - stats["aoa_mean"]) / stats["aoa_std"]
    aoa_rad = np.deg2rad(np.float32(aoa_degrees))
    return np.array(
        [aoa_norm, np.sin(aoa_rad), np.cos(aoa_rad)],
        dtype=np.float32,
    )


class TransolverPointFieldSurrogate(nn.Module):
    """Point-cloud → per-point scalar prediction via Transolver++."""

    def __init__(
        self,
        feature_dim: int = 6,
        out_dim: int = 1,
        n_hidden: int = 96,
        n_layers: int = 4,
        n_head: int = 4,
        slice_num: int = 16,
        mlp_ratio: int = 2,
        dropout: float = 0.05,
    ) -> None:
        super().__init__()
        self.backbone = TransolverPlusModel(
            space_dim=feature_dim, fun_dim=0, out_dim=out_dim,
            n_hidden=n_hidden, n_layers=n_layers, n_head=n_head,
            mlp_ratio=mlp_ratio, slice_num=slice_num,
            dropout=dropout, unified_pos=False,
        )

    def forward(
        self,
        x: torch.Tensor,
        pos: torch.Tensor,
        condition: torch.Tensor,
    ) -> torch.Tensor:
        return self.backbone((x, pos, condition))


@torch.no_grad()
def predict_case(
    model: nn.Module,
    case: dict,
    stats: dict,
    chunk_points: int = CHUNK_POINTS,
) -> np.ndarray:
    """Score a full case's ~8k points in chunks. Returns denormalised (N, D)."""
    model.eval()
    torch.manual_seed(SEED)

    target_mean_t = torch.tensor(
        stats["target_mean"], dtype=torch.float32, device=device,
    )
    target_std_t = torch.tensor(
        stats["target_std"], dtype=torch.float32, device=device,
    )

    points = case["points"].astype(np.float32)
    normals = case["normals"].astype(np.float32)
    pos_all = ((points - stats["point_mean"]) / stats["point_std"]).astype(np.float32)
    x_all = np.concatenate([pos_all, normals], axis=1).astype(np.float32)

    cond = torch.tensor(
        aoa_condition(case["aoa"], stats),
        dtype=torch.float32, device=device,
    ).unsqueeze(0)

    preds: list[np.ndarray] = []
    for start in range(0, len(points), chunk_points):
        end = min(start + chunk_points, len(points))
        x = torch.tensor(x_all[start:end], dtype=torch.float32, device=device).unsqueeze(0)
        pos = torch.tensor(pos_all[start:end], dtype=torch.float32, device=device).unsqueeze(0)
        pred_norm = model(x, pos, cond)
        pred_phys = pred_norm * target_std_t + target_mean_t
        preds.append(pred_phys.squeeze(0).cpu().numpy())

    return np.concatenate(preds, axis=0)


def case_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Global MAE + RMSE on physical Cp units, plus per-component means."""
    err = y_pred - y_true
    return {
        "mae":  float(np.mean(np.abs(err))),
        "rmse": float(np.sqrt(np.mean(err ** 2))),
    }


# ---- Load the dataset from the Registry ----
# No `wandb.init` here — eval runs each `use_artifact` themselves in §4
# to record their own lineage edges. This download is just for the
# in-notebook eval loop.
api = wandb.Api()
ds_art = api.artifact(DATASET_REGISTRY_PATH)
ds_dir = Path(ds_art.download())
all_cases: list[dict] = [_load_npz_case(p) for p in sorted(ds_dir.glob("*.npz"))]

# ---- Derive AoA stats (only for aoa_condition) ----
# The point/target normalisation stats come from each checkpoint's
# embedded `stats` dict — those are the ones the model was trained on.
stats_aoa = {
    "aoa_mean": np.float32(np.mean([c["aoa"] for c in all_cases])),
    "aoa_std":  np.float32(max(np.std([c["aoa"] for c in all_cases]), 1.0)),
}

print(f"loaded {len(all_cases)} cases from Registry")
print(f"  geo_ids: {sorted(set(c['geo_id'] for c in all_cases))}")
print(f"  AoAs:    {sorted(set(c['aoa'] for c in all_cases))}")


## 3. Find the checkpoints to eval

`experiment_tracking_hilift.ipynb` produced one training run named
`hilift-transolverpp-train`. Its logged artifacts include every
checkpoint version — one per cadence epoch (10, 20, 30, 40, 50) plus
any `is_best` carve-outs. We query the run by name and iterate its
`logged_artifacts()` filtered by `type == "model"`.

This is deliberately a *pull* operation, not a push. The training
run doesn't know an eval fleet is coming; the eval fleet doesn't
need any handoff other than the run name.


In [ ]:
api = wandb.Api()

training_runs = list(api.runs(
    f"{ENTITY}/{PROJECT}",
    filters={"display_name": TRAINING_RUN_QUERY},
    order="-created_at",
))
if not training_runs:
    raise RuntimeError(
        f"No run named {TRAINING_RUN_QUERY!r} found in {ENTITY}/{PROJECT}. "
        f"Run experiment_tracking_hilift.ipynb first."
    )
training_run = training_runs[0]
training_run_id = training_run.id
print(f"training run: {training_run.name}   id: {training_run_id}")
print(f"  url: {training_run.url}")

# ---- Iterate the training run's logged model artifacts ----
checkpoints: list[dict] = []
for art in training_run.logged_artifacts():
    if art.type != "model":
        continue
    meta = art.metadata or {}
    epoch = int(meta.get("epoch", -1))
    ref = f"{ENTITY}/{PROJECT}/{art.name.split(':')[0]}:{art.version}"
    aliases = list(art.aliases or [])
    checkpoints.append({
        "artifact": art,
        "ref": ref,
        "version": art.version,
        "epoch": epoch,
        "aliases": aliases,
        "val_loss": float(meta.get("val_loss", float("nan"))),
    })

checkpoints.sort(key=lambda c: c["epoch"])
print()
print(f"found {len(checkpoints)} checkpoint(s) to eval:")
for c in checkpoints:
    alias_str = ", ".join(c["aliases"]) or "-"
    print(
        f"  epoch {c['epoch']:>3}  version {c['version']:>4}  "
        f"val_loss={c['val_loss']:.4f}  aliases=[{alias_str}]"
    )


## 4. Async eval — one run per checkpoint

Each iteration below is the shape of a standalone eval script. In
production they would launch on separate machines, triggered by the
checkpoint artifact appearing on the backend (W&B Automations,
GitHub Actions, Argo, Slurm — anything). Here they run sequentially
inside one kernel; **the API is identical either way**.

Each eval run:

- Opens its own `wandb.init(job_type="eval", group=training_run_id,
  resume="never")` — never touches the training run.
- Reads the checkpoint's metadata to learn the epoch.
- `use_artifact(...)` on the checkpoint to create the input lineage
  edge (eval run → checkpoint artifact).
- Rebuilds `TransolverPointFieldSurrogate` and loads the state_dict.
- Runs the offline eval battery on all 12 cases: global MAE / RMSE
  on Cp; per-AoA buckets (4 values); per-geometry buckets
  (`LHC001`, `LHC002`, `LHC003`); wall-time latency per case.
- Uploads a nested `eval-results-{training_run_id}` artifact with
  the full per-case metrics for auditability.
- Writes flat `eval/*` scalars into its own summary.

Full 12 cases are evaluated (rather than just the training
notebook's `test_cases`) so per-AoA and per-geometry buckets have
enough samples to be meaningful.


In [ ]:
def eval_checkpoint_full(
    model: nn.Module,
    cases: list[dict],
    stats: dict,
) -> dict:
    """Score every case; return flat eval/* dict + a per-case payload."""
    per_case: list[dict] = []
    t_total = 0.0

    for case in cases:
        t0 = time.perf_counter()
        pred = predict_case(model, case, stats)
        t_total += (time.perf_counter() - t0)

        m = case_metrics(case["target"], pred)
        per_case.append({
            "run_id":  case["run_id"],
            "geo_id":  case["geo_id"],
            "aoa":     float(case["aoa"]),
            "mae":     m["mae"],
            "rmse":    m["rmse"],
            "n_points": int(case["points"].shape[0]),
        })

    # ---- Global aggregates ----
    maes  = np.array([c["mae"]  for c in per_case])
    rmses = np.array([c["rmse"] for c in per_case])
    flat: dict = {
        "eval/mae":                   float(np.mean(maes)),
        "eval/rmse":                  float(np.mean(rmses)),
        "eval/n_cases":               len(per_case),
        "eval/latency_ms_per_case":   (t_total / max(1, len(per_case))) * 1000.0,
    }

    # ---- Per-AoA buckets ----
    for aoa_val in sorted({c["aoa"] for c in per_case}):
        rows = [c for c in per_case if c["aoa"] == aoa_val]
        key = f"eval/aoa_{int(aoa_val):02d}"
        flat[f"{key}_mae"]  = float(np.mean([c["mae"]  for c in rows]))
        flat[f"{key}_rmse"] = float(np.mean([c["rmse"] for c in rows]))

    # ---- Per-geometry buckets ----
    for geo in sorted({c["geo_id"] for c in per_case}):
        rows = [c for c in per_case if c["geo_id"] == geo]
        key = f"eval/geo_{geo}"
        flat[f"{key}_mae"]  = float(np.mean([c["mae"]  for c in rows]))
        flat[f"{key}_rmse"] = float(np.mean([c["rmse"] for c in rows]))

    return {"flat": flat, "per_case": per_case}


# ==== Run one eval job per checkpoint =================================
for ckpt_info in checkpoints:
    ep = ckpt_info["epoch"]
    ref = ckpt_info["ref"]

    with wandb.init(
        entity=ENTITY, project=PROJECT,
        name=f"eval-{TRAINING_RUN_QUERY}-ep{ep:03d}",
        group=training_run_id,
        job_type="eval",
        resume="never",
        config={
            "job_type":                        "eval",
            "source_training_run_id":          training_run_id,
            "source_training_run_name":        TRAINING_RUN_QUERY,
            "source_checkpoint_artifact":      ref,
            "source_checkpoint_version":       ckpt_info["version"],
            "source_checkpoint_epoch":         ep,
            "source_checkpoint_aliases":       ckpt_info["aliases"],
            "eval_dataset":                    DATASET_REGISTRY_PATH,
            "eval_split":                      "all-cases",
            "eval_code_version":               "v1.0.0",
            "eval_timestamp":                  utc_ts(),
        },
        tags=["notebook", "advanced", "async-eval", "hilift", "eval"],
    ) as eval_run:
        # eval_run --uses--> checkpoint artifact (input lineage edge)
        input_art = eval_run.use_artifact(ref, type="model")
        d = Path(input_art.download())
        ckpt = torch.load(
            next(d.glob("*.pt")),
            map_location=device,
            weights_only=False,
        )

        # Rebuild the model + load state_dict.
        arch = ckpt.get("arch", {})
        model_kwargs = {
            "feature_dim": 6,
            "out_dim":     int(ckpt.get("target_dim", 1)),
            "n_hidden":    arch.get("n_hidden",  96),
            "n_layers":    arch.get("n_layers",   4),
            "n_head":      arch.get("n_head",     4),
            "slice_num":   arch.get("slice_num", 16),
            "mlp_ratio":   arch.get("mlp_ratio",  2),
        }
        eval_model = TransolverPointFieldSurrogate(**model_kwargs).to(device)
        eval_model.load_state_dict(ckpt["state_dict"])

        # Merge AoA stats with the checkpoint's point/target stats so
        # predict_case can run.
        ckpt_stats = dict(ckpt["stats"])
        ckpt_stats.setdefault("aoa_mean", stats_aoa["aoa_mean"])
        ckpt_stats.setdefault("aoa_std",  stats_aoa["aoa_std"])

        # ---- Run the eval battery ----
        result = eval_checkpoint_full(eval_model, all_cases, ckpt_stats)
        flat = result["flat"]
        per_case = result["per_case"]

        # ---- Chart-friendly flat scalars ----
        eval_run.log({**flat, "epoch": ep, "eval/checkpoint_epoch": ep})

        # ---- Namespaced summary (used by the aggregator in §5) ----
        eval_run.summary.update({
            **flat,
            "eval/checkpoint_epoch":    ep,
            "eval/checkpoint_artifact": ref,
        })

        # ---- Full nested eval-results artifact for auditability ----
        results_path = TMP_DIR / f"eval_ep{ep:03d}.json"
        results_path.write_text(json.dumps({
            "source_training_run_id": training_run_id,
            "source_checkpoint_artifact": ref,
            "checkpoint_epoch": ep,
            "eval_run_id": eval_run.id,
            "eval_dataset": DATASET_REGISTRY_PATH,
            "flat_metrics": flat,
            "per_case": per_case,
            "evaluated_at": utc_ts(),
        }, indent=2))
        results_art = wandb.Artifact(
            name=f"eval-results-{training_run_id}",
            type="eval-results",
            metadata={
                "source_training_run_id": training_run_id,
                "source_checkpoint_artifact": ref,
                "checkpoint_epoch": ep,
                "eval_run_id": eval_run.id,
                "metrics_schema_version": "1.0",
                "created_at": utc_ts(),
            },
        )
        results_art.add_file(str(results_path), name="eval_metrics.json")
        eval_run.log_artifact(results_art)

        print(
            f"eval ep{ep:>3}: mae={flat['eval/mae']:.4f}   "
            f"rmse={flat['eval/rmse']:.4f}   "
            f"latency={flat['eval/latency_ms_per_case']:.1f} ms/case   "
            f"run_id={eval_run.id}"
        )


## 5. Aggregate — merge eval results into the training run summary

Post-hoc aggregation. Uses `wandb.Api().runs(...)` with a server-side
filter on `config.source_training_run_id` to find every eval run tied
to this training run — no external state, no directory scanning.
Computes best (min `eval/rmse`) and latest (max
`eval/checkpoint_epoch`) and writes compact keys back to the
training run via `api.run(...).summary.update({...})`. All keys are
namespaced under `eval/*` — nothing training-owned is touched.


In [ ]:
api = wandb.Api()
eval_runs = list(api.runs(
    f"{ENTITY}/{PROJECT}",
    filters={
        "config.source_training_run_id": training_run_id,
        "config.job_type": "eval",
    },
))
print(f"found {len(eval_runs)} eval run(s) for training_run_id={training_run_id}")

records: list[dict] = []
for r in eval_runs:
    s = dict(r.summary)
    c = dict(r.config)
    ep = s.get("eval/checkpoint_epoch") or c.get("source_checkpoint_epoch")
    rmse = s.get("eval/rmse")
    if ep is None or rmse is None:
        print(f"  [skip] {r.id} — missing eval/rmse or checkpoint epoch")
        continue
    records.append({
        "run_id":              r.id,
        "epoch":               int(ep),
        "mae":                 float(s.get("eval/mae", float("nan"))),
        "rmse":                float(rmse),
        "latency_ms_per_case": float(s.get("eval/latency_ms_per_case", float("nan"))),
        "checkpoint_artifact": s.get("eval/checkpoint_artifact")
                                or c.get("source_checkpoint_artifact"),
    })

print()
print("eval records (sorted by epoch):")
for r in sorted(records, key=lambda x: x["epoch"]):
    print(
        f"  epoch {r['epoch']:>3}  mae={r['mae']:.4f}  rmse={r['rmse']:.4f}  "
        f"run_id={r['run_id']}"
    )

if not records:
    raise RuntimeError("No eval runs found to aggregate — did §4 complete?")

best   = min(records, key=lambda r: r["rmse"])
latest = max(records, key=lambda r: r["epoch"])

agg = {
    "eval/best/mae":                    best["mae"],
    "eval/best/rmse":                   best["rmse"],
    "eval/best/checkpoint_epoch":       best["epoch"],
    "eval/best/checkpoint_artifact":    best["checkpoint_artifact"],
    "eval/best/run_id":                 best["run_id"],
    "eval/latest/mae":                  latest["mae"],
    "eval/latest/rmse":                 latest["rmse"],
    "eval/latest/checkpoint_epoch":     latest["epoch"],
    "eval/latest/checkpoint_artifact":  latest["checkpoint_artifact"],
    "eval/latest/run_id":               latest["run_id"],
    "eval/history_count":               len(records),
    "eval/aggregated_at":               utc_ts(),
    "eval/run_ids":                     [r["run_id"] for r in records],
}
print()
print("computed aggregate summary:")
for k, v in agg.items():
    if not isinstance(v, list):
        print(f"  {k}: {v}")

training_run_ref = api.run(f"{ENTITY}/{PROJECT}/{training_run_id}")
training_run_ref.summary.update(agg)
training_run_ref.update()
print()
print(f"wrote {len(agg)} keys under eval/* into training run {training_run_id}")
print(f"training run URL: {training_run_ref.url}")


## 6. Why not resume?

Reopening the training run from the eval process
(`wandb.init(id=train_run_id, resume="allow")`) sounds convenient
but breaks the run's data model:

- **Config overwrites.** Any config key the eval process writes
  merges silently on the server. If eval and training share a key
  by accident, the training value is lost — no conflict detection.
- **Step interleaving.** `run.log()` writes are append-only per
  run. Concurrent writers produce interleaved, non-monotonic step
  sequences; the resulting charts are meaningless.
- **`run.finish()` races.** Whichever process calls `finish()`
  first marks the run as finished. Subsequent logs from the other
  process go into a "finished" run.
- **Summary last-write-wins.** `run.summary` is a flat dict.
  Concurrent writes to different keys are OK in principle but
  order-dependent in practice.

None of these show up in a demo where you run everything in one
kernel — they show up in production when a training job takes 12
hours and eval workers spawn on a schedule. The multi-run + group
pattern above avoids every one of them structurally: each writer
owns its own run.

Long-form treatment:
`notebooks/advanced/asynchronous_evals/ARCHITECTURE.md`.


## 7. Production caveats — and how does this generalise

- **What "async" means outside the notebook.** Each eval-run
  iteration above is the shape of a standalone script. In
  production spawn N of them on separate machines, triggered by
  the checkpoint artifact appearing (W&B Automations, GitHub
  Actions, Argo, Slurm — anything). They don't coordinate; they
  don't know about each other.
- **When to aggregate.** Aggregation is safe *after* the training
  run has called `run.finish()`, or at any time provided you only
  write `eval/*` keys (which the training script never writes).
  Never write training-owned keys from the aggregator.

### How does this generalise?

**Multiple teams.** The eval battery is parameterised on
`TRAINING_RUN_QUERY`. Each team runs its own eval fleet against
its own training run — no coordination beyond the run name and
the shared `hilift` Registry. Team-scoped aliases on the eval-
results artifact (`eval-results-{team}`) keep the outputs tidy.

**Multiple architectures.** The eval helper reads the model
architecture from the checkpoint's `arch` metadata dict and
rebuilds `TransolverPointFieldSurrogate` with those kwargs. A
sweep that produced checkpoints under different `n_hidden` /
`n_layers` / `slice_num` all get evaluated by the same eval
script — the reconstruction is data-driven.

**Multiple experiments.** The aggregation script generalises
naturally: loop over a list of `training_run_id`s (from a Sweep,
from a scheduled run pool, from a Registry Collection) and produce
a comparison DataFrame with one row per training run and columns
for `eval/best/*`. Pushing that back into a Report gives a
cross-experiment leaderboard that stays live as new experiments
land.


## 8. Wrap up — what to look at in the W&B UI

**Group view.** Filter runs by `group=<training_run_id>` (printed
in §3). One `hilift-transolverpp-train` run and N `eval-*-epNNN`
runs appear in one table.

**Lineage graph.** For each checkpoint version
`hilift_transolverpp_ckpt:vN`:
- Output edge from the training run (produced it).
- Input edge into its eval run (via `use_artifact`).
- Each eval run also produces an `eval-results-{training_run_id}`
  artifact.

**Training run summary panel.** Now shows both training-owned keys
(`train/*`, `best_epoch`, `best_val_loss`, checkpoint metadata) and
aggregated eval keys (`eval/best/*`, `eval/latest/*`,
`eval/history_count`, `eval/run_ids`) — no collision because the
eval keys are all namespaced.

**Next steps.** With the eval fleet running, you can:
- Compare per-AoA and per-geometry buckets across checkpoints
  in the eval runs' workspace — which cases the model struggles
  on, which epochs improve them.
- Add the training run's `eval/best/*` summary keys as columns to
  the workshop's Report (`reports.ipynb`) so the Report reflects
  actual held-out performance, not just training-time val.
